# Day 06 · Classification: from yes/no to probabilities

**Goal:** build logistic regression from scratch (sigmoid + log-loss + gradient descent), then use scikit-learn to predict **campus placement**, draw the **decision boundary**, tune the **threshold** to a business cost, and handle **multiclass** with softmax vs one-vs-rest.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`.

**Runtime:** CPU is fine. Every cell finishes in seconds.

| Section | What you'll do |
|---|---|
| 1 | A placement dataset (and why a straight line fails) |
| 2 | Sigmoid, log-loss, gradient descent, by hand |
| 3 | The same model in scikit-learn; reading odds ratios |
| 4 | Decision boundary in 2-D |
| 5 | Thresholds: precision vs recall vs rupees |
| 6 | Multiclass: softmax vs one-vs-rest on Iris |
| 7 | 🎯 Try it yourself |
| 8 | Homework starter + key takeaways |

## 0. Setup

Nothing to install; everything ships with Colab.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, log_loss

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams["figure.figsize"] = (8, 4.5)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 1. A campus-placement dataset 🎓

**What:** 600 synthetic students with three features: `cgpa`, `aptitude` (test score /100), `internships` (count). Label `placed` = 1/0. It is synthetic so that it is reproducible and so that we *know* the true relationship; the shapes are realistic (placement probability rises smoothly with CGPA, not as a cliff).

**Why start with "why not linear regression?":** students always ask. Fitting a line to 0/1 labels gives predictions like 1.3 and −0.2, and a single outlier moves the cut-off for everyone. Look at the plot.

In [ ]:
n = 600
cgpa = np.clip(rng.normal(7.2, 1.0, n), 5.0, 9.9).round(2)
aptitude = np.clip(rng.normal(60, 15, n), 20, 99).round(0)
internships = rng.choice([0, 1, 2, 3], size=n, p=[0.35, 0.35, 0.2, 0.1])

# the TRUE (hidden) rule: log-odds is linear in the features
logit = 1.8 * (cgpa - 7.0) + 0.05 * (aptitude - 60) + 0.7 * internships - 0.6
p_true = 1 / (1 + np.exp(-logit))
placed = (rng.uniform(size=n) < p_true).astype(int)

df = pd.DataFrame({"cgpa": cgpa, "aptitude": aptitude, "internships": internships, "placed": placed})
print(df.head(), "\n\nplacement rate:", df.placed.mean().round(3))
X, y = df[["cgpa", "aptitude", "internships"]], df["placed"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

In [ ]:
# Why not linear regression on 0/1 labels? Fit it on CGPA only and look.
lin = LinearRegression().fit(X_tr[["cgpa"]], y_tr)
grid = np.linspace(5, 10, 200)[:, None]
plt.scatter(X_tr.cgpa, y_tr + rng.normal(0, .02, len(y_tr)), s=12, alpha=.5, label="students (0 = not placed, 1 = placed)")
plt.plot(grid, lin.predict(grid), c="#FF6B8A", lw=2, label="linear regression")
plt.axhline(0, c="k", lw=.5); plt.axhline(1, c="k", lw=.5)
plt.xlabel("CGPA"); plt.ylabel("placed"); plt.legend(); plt.title("Predictions above 1 and below 0: not probabilities")
plt.show()
print("prediction for CGPA 9.9:", lin.predict([[9.9]]).round(2), "  for CGPA 5.0:", lin.predict([[5.0]]).round(2))

## 2. Logistic regression from scratch ✍️

Three functions, each a few lines:

1. **sigmoid** $\sigma(z) = \dfrac{1}{1+e^{-z}}$ squashes any score into (0, 1).
2. **log-loss** $J = -\frac1m\sum\big[y\log p + (1-y)\log(1-p)\big]$ punishes confident mistakes.
3. **gradient** $\nabla J = \frac1m X^\top(p - y)$ — *the same formula as linear regression*, only $p$ went through the sigmoid.

We standardise features first (gradient descent converges far faster when features share a scale).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)                       # never log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def fit_logreg(X, y, lr=0.1, epochs=3000):
    Xb = np.c_[np.ones(len(X)), X]                     # bias column of 1s
    theta = np.zeros(Xb.shape[1])
    history = []
    for _ in range(epochs):
        p = sigmoid(Xb @ theta)
        grad = Xb.T @ (p - y) / len(y)                 # (1/m) Xᵀ(p − y)
        theta -= lr * grad
        history.append(bce(y, p))
    return theta, history

scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)
theta, hist = fit_logreg(Xs_tr, y_tr.values)

p_te = sigmoid(np.c_[np.ones(len(Xs_te)), Xs_te] @ theta)
print("theta (bias, cgpa, aptitude, internships):", theta.round(3))
print(f"test log-loss {bce(y_te.values, p_te):.3f} | test accuracy {((p_te >= .5) == y_te.values).mean():.3f}")
plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("log-loss"); plt.title("Gradient descent on log-loss (convex: one smooth descent)"); plt.show()

## 3. Same model in scikit-learn, and how to *read* it 📖

`LogisticRegression` minimises the same log-loss (plus a small L2 penalty controlled by `C`; **bigger C = weaker regularisation**, the opposite of Ridge's `alpha`). Because the penalty is tiny by default, its weights should land very close to our hand-rolled ones.

The payoff of logistic regression is interpretability: a weight $\theta_j$ means *one unit of feature j multiplies the odds of placement by $e^{\theta_j}$*.

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X_tr, y_tr)
lr_model = clf[-1]
print("sklearn weights:", lr_model.coef_.round(3), " bias:", lr_model.intercept_.round(3))
print("ours           :", theta[1:].round(3), " bias:", theta[:1].round(3))
print("test accuracy  :", round(clf.score(X_te, y_te), 3))

# odds ratios on the ORIGINAL scale: refit without the scaler so units are CGPA points / score points / internships
raw = LogisticRegression(C=1.0, max_iter=2000).fit(X_tr, y_tr)
odds = pd.Series(np.exp(raw.coef_[0]), index=X.columns).round(2)
print("\nOdds multiplier per +1 unit of each feature:\n", odds)
print("\nRead: '+1 internship multiplies the odds of placement by", odds['internships'], "'. (True generator used e^0.7 ≈ 2.0)")

In [ ]:
# The three outputs you will use constantly:
proba  = clf.predict_proba(X_te)[:, 1]     # P(placed)  — column order = clf.classes_
scores = clf.decision_function(X_te)       # raw log-odds z; sigmoid(scores) == proba
labels = clf.predict(X_te)                 # proba >= 0.5

print("classes_       :", lr_model.classes_)
print("first 5 proba  :", proba[:5].round(3))
print("first 5 z      :", scores[:5].round(2), " -> sigmoid:", sigmoid(scores[:5]).round(3))
print("first 5 labels :", labels[:5])
print("test log-loss  :", round(log_loss(y_te, proba), 3))

## 4. The decision boundary, drawn 🖊️

With two features the boundary $\theta_0 + \theta_1 x_1 + \theta_2 x_2 = 0$ is a straight line. We refit on `cgpa` and `aptitude` only, so we can draw it. `DecisionBoundaryDisplay` works for any 2-feature classifier; with `response_method="predict_proba"` it shades by probability, so you *see* uncertainty near the line.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

clf2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr[["cgpa", "aptitude"]], y_tr)

fig, ax = plt.subplots(figsize=(8, 5.5))
DecisionBoundaryDisplay.from_estimator(clf2, X_tr[["cgpa", "aptitude"]], response_method="predict_proba",
                                       cmap="RdYlGn", alpha=.45, ax=ax, xlabel="CGPA", ylabel="aptitude score")
ax.scatter(X_tr.cgpa, X_tr.aptitude, c=y_tr, cmap="RdYlGn", edgecolor="k", s=22)
ax.set_title("P(placed). The 0.5 contour is the decision boundary — a straight line")
plt.show()
print("2-feature test accuracy:", round(clf2.score(X_te[["cgpa", "aptitude"]], y_te), 3))

## 5. Thresholds: the business decides, not the maths ⚖️

`predict()` uses 0.5. Nothing forces that. Lowering the threshold predicts "placed" more often → **recall** rises (fewer real positives missed) but **precision** falls (more false alarms). Raising it does the reverse.

Which is right depends on the **cost** of each mistake. Suppose the placement cell runs a paid 2-day interview bootcamp for everyone the model flags as "likely placed":
* a **false positive** wastes a ₹1,500 seat,
* a **false negative** (a placeable student not coached) is estimated to cost ₹6,000 in a lost offer.

Let's find the threshold that minimises rupees, not the one that maximises accuracy.

In [ ]:
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    y_hat = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, y_hat).ravel()
    rows.append({"threshold": round(t, 2), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "accuracy": accuracy_score(y_te, y_hat),
                 "precision": precision_score(y_te, y_hat, zero_division=0),
                 "recall": recall_score(y_te, y_hat),
                 "cost_₹": 1500 * fp + 6000 * fn})
tab = pd.DataFrame(rows).set_index("threshold")
print(tab.round(3))
best = tab["cost_₹"].idxmin()
print(f"\nMax accuracy at t = {tab['accuracy'].idxmax()}, but MIN COST at t = {best} (₹{tab.loc[best, 'cost_₹']:,}).")

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(tab.index, tab.precision, marker="o", label="precision", c="#4FD3F2")
ax1.plot(tab.index, tab.recall, marker="o", label="recall", c="#FFB43A")
ax1.set_xlabel("threshold"); ax1.set_ylabel("precision / recall"); ax1.legend(loc="center left")
ax2 = ax1.twinx(); ax2.plot(tab.index, tab["cost_₹"], c="#FF6B8A", lw=2, label="cost (₹)"); ax2.set_ylabel("total cost ₹", color="#FF6B8A")
ax2.axvline(best, ls="--", c="#FF6B8A", lw=1)
plt.title("Precision falls, recall rises, cost has a minimum. Pick the minimum."); plt.show()

> 💼 **Career note:** "we chose the threshold by minimising expected business cost on the validation set" is a sentence that separates a fresher who ran `.predict()` from one who thinks like a data scientist. Say it in interviews. Day 12 adds ROC/PR curves to make this rigorous.

## 6. Multiclass: softmax vs one-vs-rest 🌸

For $K$ classes there are two routes:
* **Softmax (multinomial):** one model, $K$ score rows, probabilities $p_k = e^{z_k}/\sum_j e^{z_j}$ that sum to 1. This is sklearn's default with the `lbfgs` solver and is the output layer of every neural-net classifier.
* **One-vs-rest (OvR):** $K$ separate binary models ("setosa vs not", …); pick the most confident. `OneVsRestClassifier` wraps any binary model.

Iris: 150 flowers, 4 measurements, 3 species.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.multiclass import OneVsRestClassifier

iris = load_iris(as_frame=True)
Xi, yi = iris.data, iris.target
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=SEED, stratify=yi)

softmax_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xi_tr, yi_tr)
ovr_clf = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xi_tr, yi_tr)

print("softmax coef shape:", softmax_clf[-1].coef_.shape, "(one weight row per class)")
print("softmax accuracy  :", round(softmax_clf.score(Xi_te, yi_te), 3))
print("OvR accuracy      :", round(ovr_clf.score(Xi_te, yi_te), 3))
print("\nsame flower, two probability vectors:")
print(" softmax:", softmax_clf.predict_proba(Xi_te.iloc[:1]).round(3), "sums to", softmax_clf.predict_proba(Xi_te.iloc[:1]).sum().round(3))
print(" OvR    :", ovr_clf.predict_proba(Xi_te.iloc[:1]).round(3), "(3 sigmoids, renormalised)")

In [ ]:
# Softmax by hand: the 3 raw scores for one flower -> probabilities
z = softmax_clf.decision_function(Xi_te.iloc[:1])[0]
p = np.exp(z - z.max()) / np.exp(z - z.max()).sum()      # subtract max for numerical safety; result identical
print("raw scores z:", z.round(2))
print("softmax(z)  :", p.round(3), "| matches predict_proba:", np.allclose(p, softmax_clf.predict_proba(Xi_te.iloc[:1])[0]))
print("predicted   :", iris.target_names[p.argmax()], "| true:", iris.target_names[yi_te.iloc[0]])

## 7. 🎯 Try it yourself: regularisation strength `C` (10 min)

Train the placement model with `C` in `[0.001, 0.01, 0.1, 1, 10, 100]` (scaler in the pipeline). For each, print the **largest |coefficient|** and the **test log-loss**.

Questions: (a) which direction makes weights shrink? (b) does the tiny-C model under- or over-fit? (c) why is it harmless here but would matter with 500 features?

In [ ]:
# YOUR CODE HERE
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    # build pipeline with LogisticRegression(C=C, max_iter=1000), fit on X_tr/y_tr
    # print C, max |coef|, log_loss on X_te
    pass

<details><summary>Solution</summary>

```python
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000)).fit(X_tr, y_tr)
    ll = log_loss(y_te, m.predict_proba(X_te)[:, 1])
    print(f"C={C:<7} max|coef|={np.abs(m[-1].coef_).max():.3f}  test log-loss={ll:.3f}")
```
(a) Smaller C → stronger penalty → smaller weights. (b) C = 0.001 underfits: all weights ≈ 0, probabilities ≈ the base rate, log-loss near 0.69. (c) With 3 well-behaved features there is little to overfit; with 500 features (text!) the penalty is what keeps the model sane. Pick C by cross-validation (`LogisticRegressionCV`).
</details>

## Homework starter 📬

**Medium task:** Telco customer churn (7,043 customers; real IBM sample data). Fit a scaled logistic regression on numeric features + `Contract`, then find the threshold that minimises cost if a **retention call costs ₹200** and a **lost customer costs ₹3,000**.

The scaffold loads and cleans the data and fits the model. You write the cost loop (copy Section 5).

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    telco = pd.read_csv(URL)
    telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # 11 blanks -> NaN
    telco = telco.dropna(subset=["TotalCharges"])
    telco["churn"] = (telco["Churn"] == "Yes").astype(int)
    feats = pd.get_dummies(telco[["tenure", "MonthlyCharges", "TotalCharges", "Contract"]], drop_first=True, dtype=float)
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(feats, telco["churn"], test_size=0.25, random_state=SEED, stratify=telco["churn"])
    churn_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_tr, yc_tr)
    churn_proba = churn_clf.predict_proba(Xc_te)[:, 1]
    print("churn rate:", yc_tr.mean().round(3), "| test accuracy at 0.5:", round(churn_clf.score(Xc_te, yc_te), 3))
    print("odds ratios:\n", pd.Series(np.exp(churn_clf[-1].coef_[0]), index=feats.columns).round(2))
    # YOUR CODE HERE: loop thresholds 0.05..0.95, cost = 200*FP + 3000*FN, print the best threshold
except Exception as e:
    print("Could not download the CSV (offline?). Skipping homework starter:", e)

## Key takeaways ✅

* **Logistic regression = linear score → sigmoid → probability.** The log-odds are linear in the features; $e^{\theta_j}$ is an odds ratio you can explain to a bank.
* **Log-loss**, not MSE: convex, punishes confident mistakes, equals maximum likelihood. Its gradient $(1/m)X^\top(p-y)$ has the same shape as linear regression's.
* **The decision boundary is a straight line** in the raw features (curve it with polynomial features).
* **The threshold is a business decision.** Tune it to the cost of FP vs FN; 0.5 is only a default.
* **`C` is inverse regularisation** (small C = strong penalty). Scale features; raise `max_iter` if warned.
* **Multiclass:** softmax (one model, probabilities sum to 1; sklearn's default) or one-vs-rest (K binary models).

**Tomorrow (Day 07):** two models that could not be more different from today's: **KNN** (no training, just neighbours) and **Naive Bayes** (probabilities by counting), and a real spam filter.